In [76]:
import numpy as np
from astropy.io import fits

In [77]:
opencov = fits.open('../data/covariance/xip_xim_map3_covariance_8ARCMINCUT_31Oct24.fits')
cov_matrix = opencov[1].data
opendata_2pt = fits.open('../data/dv/sim_2pt-NLA-cosmoCosmogrid-04Nov24.fits')
opendata_map3 = fits.open('../data/dv/sim_map3-cosmoCosmogrid-6Nov24-8arcmincut-WITH_COV.fits')

data_vector = np.zeros(480)
for i in range(480):
    if i < 200:
        data_vector[i] = opendata_2pt[2].data[i][3]
    elif i < 400:
        data_vector[i] = opendata_2pt[3].data[i-200][3]
    else:
        data_vector[i] = opendata_map3[1].data[i-400][6]

In [83]:
print(opendata_map3[1].data)

[(1, 1, 1,  7.,  7.,  7., 1.91966997e-10)
 (1, 1, 1, 14., 14., 14., 8.79050074e-11)
 (1, 1, 1, 25., 25., 25., 3.82239005e-11)
 (1, 1, 1, 40., 40., 40., 2.87543577e-12)
 (1, 1, 2,  7.,  7.,  7., 2.75008092e-10)
 (1, 1, 2, 14., 14., 14., 1.23340690e-10)
 (1, 1, 2, 25., 25., 25., 5.40051862e-11)
 (1, 1, 2, 40., 40., 40., 4.10267890e-12)
 (1, 1, 3,  7.,  7.,  7., 3.28840819e-10)
 (1, 1, 3, 14., 14., 14., 1.45769724e-10)
 (1, 1, 3, 25., 25., 25., 6.41676389e-11)
 (1, 1, 3, 40., 40., 40., 4.90122368e-12)
 (1, 1, 4,  7.,  7.,  7., 3.44537175e-10)
 (1, 1, 4, 14., 14., 14., 1.52231244e-10)
 (1, 1, 4, 25., 25., 25., 6.71338753e-11)
 (1, 1, 4, 40., 40., 40., 5.13568732e-12)
 (1, 2, 2,  7.,  7.,  7., 4.14235735e-10)
 (1, 2, 2, 14., 14., 14., 1.81393529e-10)
 (1, 2, 2, 25., 25., 25., 8.02333216e-11)
 (1, 2, 2, 40., 40., 40., 6.16253162e-12)
 (1, 2, 3,  7.,  7.,  7., 5.12037169e-10)
 (1, 2, 3, 14., 14., 14., 2.21066516e-10)
 (1, 2, 3, 25., 25., 25., 9.85917972e-11)
 (1, 2, 3, 40., 40., 40., 7.621672

In [35]:
# Parameters
n_noisy_samples = 1000000  # Number of noisy samples to generate

# Generate 2000 noisy data vectors
noisy_data_vectors = np.random.multivariate_normal(mean=data_vector, cov=cov_matrix, size=n_noisy_samples)

# Compute the empirical covariance matrix of the generated noisy data vectors
empirical_cov_matrix = np.cov(noisy_data_vectors, rowvar=False)

# Calculate the normalized Frobenius norm of the difference between covariances
frobenius_diff = np.linalg.norm(empirical_cov_matrix - cov_matrix, ord='fro') / np.linalg.norm(cov_matrix, ord='fro')

# Output the results
print("Normalized Frobenius difference between empirical and original covariance matrices:", frobenius_diff)

# Optional: Threshold for similarity (you may adjust this threshold)
threshold = 0.0  # Example threshold for similarity (10% difference)
if frobenius_diff < threshold:
    print("The empirical covariance matrix is similar to the original covariance matrix.")
else:
    print("The empirical covariance matrix is significantly different from the original covariance matrix.")

Normalized Frobenius difference between empirical and original covariance matrices: 0.006265190209463862
The empirical covariance matrix is similar to the original covariance matrix.


In [42]:
#Generate the first noisy realization

noisy_data_vector = np.random.multivariate_normal(mean=data_vector, cov=cov_matrix, size=10)
print(np.shape(noisy_data_vector))

(10, 480)


In [43]:
def save_noisy_realization(number, noisy_dvs):
    opendata_2pt = fits.open('../data/dv/sim_2pt-NLA-cosmoCosmogrid-04Nov24.fits')
    opendata_2pt_500simcov = fits.open('../data/dv/sim_2pt-NLA-cosmoCosmogrid-04Nov24_500simcov.fits')
    opendata_map3 = fits.open('../data/dv/sim_map3-cosmoCosmogrid-6Nov24-8arcmincut-WITH_COV.fits')
 
    data_vector = np.zeros(480)
    for i in range(480):
        if i < 200:
            opendata_2pt[2].data[i][3] = noisy_data_vector[number][i]
            opendata_2pt_500simcov[2].data[i][3] = noisy_data_vector[number][i]
        elif i < 400:
            opendata_2pt[3].data[i-200][3] = noisy_data_vector[number][i]
            opendata_2pt_500simcov[3].data[i-200][3] = noisy_data_vector[number][i]
        else:
            opendata_map3[1].data[i-400][6] = noisy_data_vector[number][i]
        
    noisy_2pt = '/Users/gchgomes/3pcf_integrator/data/dv/noisy_realizations/sim_2pt-NLA-cosmoCosmogrid-anacov-00'+str(number+1)+'.fits'
    opendata_2pt.writeto(noisy_2pt)   

    noisy_2pt_500simcov = '/Users/gchgomes/3pcf_integrator/data/dv/noisy_realizations/sim_2pt-NLA-cosmoCosmogrid-500simcov-00'+str(number+1)+'.fits'
    opendata_2pt_500simcov.writeto(noisy_2pt_500simcov) 

    noisy_map3 = '/Users/gchgomes/3pcf_integrator/data/dv/noisy_realizations/sim_map3-NLA-cosmoCosmogrid-00'+str(number+1)+'.fits'
    opendata_map3.writeto(noisy_map3) 

In [79]:
#Now we geneare noisy dv with the compressed 2pt function

transform = np.loadtxt("../data/moped/moped-moped-compress-2pt.txt")
dv2pt = data_vector[:400]
scale_cuts_xip = np.array([20,21,22,23,40,41,42,43,60,61,62,80,81,82,83,100,101,102,103,120,121,122,123,140,141,142,143,160,161,162,163,180,181,182])
scale_cuts_xim = np.concatenate((np.arange(200,210), np.arange(220,234),np.arange(240,254), np.arange(260,273), np.arange(280,294), np.arange(300,315), np.arange(320,335), np.arange(340,355), np.arange(360,375), np.arange(380,394)))
scale_cuts = np.concatenate((scale_cuts_xip, scale_cuts_xim))
scalecut_dv2pt = np.delete(dv2pt, scale_cuts)
compressed_dv = np.zeros(96)
compressed_dv[:16] = np.dot(transform.T, scalecut_dv2pt)
compressed_dv[16:] = data_vector[400:]

In [91]:
#test noisy map3 dv:
noisy_map3 = np.random.multivariate_normal(mean=data_vector[400:], cov=compressed_cov[16:,:][:,16:], size=10)
print(data_vector[400:])
print(noisy_map3)

In [101]:
#Now we generate the transformed cov

transform_joint = np.zeros((307,96))
transform_joint[:227,:16] = transform
transform_joint[227:,16:] = np.eye(80)

cov_cut = np.delete(np.delete(cov_matrix, scale_cuts, axis=0), scale_cuts, axis=1)

compressed_cov = np.dot(transform_joint.T, np.dot(cov_cut, transform_joint))

noisy_data_vectors = np.random.multivariate_normal(mean=compressed_dv, cov=compressed_cov, size=1000000)
np.shape(noisy_data_vectors)

(1000000, 96)

In [105]:
empirical_cov_matrix = np.cov(noisy_data_vectors, rowvar=False)
frobenius_diff = np.linalg.norm(empirical_cov_matrix[16:,:][:,16:] - compressed_cov[16:,:][:,16:], ord='fro') / np.linalg.norm(compressed_cov[16:,:][:,16:], ord='fro')
print(frobenius_diff)

259.9299245962791


In [88]:
#print(np.dot(transform.T, scalecut_dv2pt))
print(compressed_dv)
print(noisy_data_vectors[3])

[ 2.88484988e+01 -7.82943158e-01 -1.07136246e+00 -1.94632179e+00
  2.74459299e+00  2.34686604e+00  4.90283124e+00  1.77902325e+00
 -2.90550780e-01  1.41830653e+00  3.91877835e-01  1.21581517e+00
  6.33226921e-01  8.99923992e-02 -2.99623315e-01  3.57339053e-02
  1.91966997e-10  8.79050074e-11  3.82239005e-11  2.87543577e-12
  2.75008092e-10  1.23340690e-10  5.40051862e-11  4.10267890e-12
  3.28840819e-10  1.45769724e-10  6.41676389e-11  4.90122368e-12
  3.44537175e-10  1.52231244e-10  6.71338753e-11  5.13568732e-12
  4.14235735e-10  1.81393529e-10  8.02333216e-11  6.16253162e-12
  5.12037169e-10  2.21066516e-10  9.85917972e-11  7.62167289e-12
  5.41751683e-10  2.32958350e-10  1.04190713e-10  8.06964426e-12
  6.50055605e-10  2.76041394e-10  1.24538947e-10  9.70088974e-12
  6.94151707e-10  2.93321123e-10  1.32903001e-10  1.03767762e-11
  7.44793674e-10  3.13003954e-10  1.42601271e-10  1.11643490e-11
  6.64301728e-10  2.83113952e-10  1.27048228e-10  9.87769425e-12
  8.58084896e-10  3.59594

In [70]:
def save_noisy_compressed_realization(number, noisy_dvs):

    opendata_map3 = fits.open('../data/dv/sim_map3-cosmoCosmogrid-6Nov24-8arcmincut-WITH_COV.fits')
 
    cut_compressed_2pt = noisy_data_vector[number][:16]
    for i in range(16,96):
        opendata_map3[1].data[i-16][6] = noisy_data_vector[number][i]

    noisy_map3 = '../data/dv/noisy_realizations/sim_map3-NLA-cosmoCosmogrid-00'+str(number+1)+'.fits'
    opendata_map3.writeto(noisy_map3) 
    
    np.savetxt('../data/dv/compressed_moped/sim-2pt-noisy-compressed-00'+str(number+1)+'.txt',cut_compressed_2pt)

In [71]:
for j in range(9):
    save_noisy_compressed_realization(j,noisy_data_vectors)

In [131]:
# Step 1: Extract sub-blocks of the covariance matrix
Sigma_2pt = compressed_cov[:16, :16]
Sigma_map3 = compressed_cov[16:, 16:]
Sigma_cross = compressed_cov[:16, 16:]

# Step 2: Generate independent noisy data vectors for 2pt and map3 parts
# Generate 10 noisy realizations for each part
noisy_2pt = np.random.multivariate_normal(compressed_dv[:16], Sigma_2pt, size=1000000)
noisy_map3 = np.random.multivariate_normal(compressed_dv[16:], Sigma_map3, size=1000000)

# Step 3: Initialize an array to store the adjusted noisy data vectors
noisy_data_vectors = []

# Loop over each generated noisy realization to apply cross-covariance adjustments
for i in range(1000000):
    # Create the correction factor for the 2pt part using Sigma_cross
    #cross_adjustment_2pt = Sigma_cross @ np.linalg.inv(Sigma_map3) @ (noisy_map3[i] - compressed_dv[16:])/2
    #noisy_2pt_adjusted = noisy_2pt[i] + cross_adjustment_2pt

    # Create the correction factor for the map3 part using Sigma_cross.T
    #cross_adjustment_map3 = Sigma_cross.T @ np.linalg.inv(Sigma_2pt) @ (noisy_2pt[i] - compressed_dv[:16])/2
    #noisy_map3_adjusted = noisy_map3[i] + cross_adjustment_map3

    # Combine the adjusted 2pt and map3 parts
    #noisy_data_vector = np.concatenate([noisy_2pt_adjusted, noisy_map3_adjusted])
    noisy_data_vector = np.concatenate([noisy_2pt[i], noisy_map3[i]])
    noisy_data_vectors.append(noisy_data_vector)

# Convert list of noisy data vectors to a numpy array
noisy_data_vector_new = np.array(noisy_data_vectors)

In [120]:
print(compressed_dv)
print(noisy_data_vector_new[3])

[ 2.88484988e+01 -7.82943158e-01 -1.07136246e+00 -1.94632179e+00
  2.74459299e+00  2.34686604e+00  4.90283124e+00  1.77902325e+00
 -2.90550780e-01  1.41830653e+00  3.91877835e-01  1.21581517e+00
  6.33226921e-01  8.99923992e-02 -2.99623315e-01  3.57339053e-02
  1.91966997e-10  8.79050074e-11  3.82239005e-11  2.87543577e-12
  2.75008092e-10  1.23340690e-10  5.40051862e-11  4.10267890e-12
  3.28840819e-10  1.45769724e-10  6.41676389e-11  4.90122368e-12
  3.44537175e-10  1.52231244e-10  6.71338753e-11  5.13568732e-12
  4.14235735e-10  1.81393529e-10  8.02333216e-11  6.16253162e-12
  5.12037169e-10  2.21066516e-10  9.85917972e-11  7.62167289e-12
  5.41751683e-10  2.32958350e-10  1.04190713e-10  8.06964426e-12
  6.50055605e-10  2.76041394e-10  1.24538947e-10  9.70088974e-12
  6.94151707e-10  2.93321123e-10  1.32903001e-10  1.03767762e-11
  7.44793674e-10  3.13003954e-10  1.42601271e-10  1.11643490e-11
  6.64301728e-10  2.83113952e-10  1.27048228e-10  9.87769425e-12
  8.58084896e-10  3.59594

In [132]:
empirical_cov_matrix = np.cov(noisy_data_vector_new, rowvar=False)
frobenius_diff = np.linalg.norm(empirical_cov_matrix - compressed_cov, ord='fro') / np.linalg.norm(compressed_cov, ord='fro')
frobenius_diff_1 = np.linalg.norm(empirical_cov_matrix[16:,:][:,16:] - compressed_cov[16:,:][:,16:], ord='fro') / np.linalg.norm(compressed_cov[16:,:][:,16:], ord='fro')
print(frobenius_diff)
print(frobenius_diff_1)

0.0036726147375579377
0.0032696575617972706


In [138]:
# Assume `covariance_matrix` is your covariance matrix (n x n)
# and `data_vector_noiseless` is your noiseless data vector (n,)

# Step 1: Cholesky decomposition
L = np.linalg.cholesky(compressed_cov)

# Step 2: Generate a single noisy data vector
def generate_noisy_vector(data_vector_noiseless, L):
    # Generate a vector of standard normal random values
    z = np.random.normal(0, 1, size=data_vector_noiseless.shape)
    # Create noise using L and z
    noise = np.dot(L, z)
    # Add noise to the noiseless data vector
    data_vector_noisy = data_vector_noiseless + noise
    return data_vector_noisy

# Generate multiple noisy vectors if needed
num_realizations = 10000  # or however many you need
noisy_data_vector_cholesky = np.array([generate_noisy_vector(compressed_dv, L) for _ in range(num_realizations)])

In [139]:
print(compressed_dv)
print(noisy_data_vector_cholesky[3])

[ 2.88484988e+01 -7.82943158e-01 -1.07136246e+00 -1.94632179e+00
  2.74459299e+00  2.34686604e+00  4.90283124e+00  1.77902325e+00
 -2.90550780e-01  1.41830653e+00  3.91877835e-01  1.21581517e+00
  6.33226921e-01  8.99923992e-02 -2.99623315e-01  3.57339053e-02
  1.91966997e-10  8.79050074e-11  3.82239005e-11  2.87543577e-12
  2.75008092e-10  1.23340690e-10  5.40051862e-11  4.10267890e-12
  3.28840819e-10  1.45769724e-10  6.41676389e-11  4.90122368e-12
  3.44537175e-10  1.52231244e-10  6.71338753e-11  5.13568732e-12
  4.14235735e-10  1.81393529e-10  8.02333216e-11  6.16253162e-12
  5.12037169e-10  2.21066516e-10  9.85917972e-11  7.62167289e-12
  5.41751683e-10  2.32958350e-10  1.04190713e-10  8.06964426e-12
  6.50055605e-10  2.76041394e-10  1.24538947e-10  9.70088974e-12
  6.94151707e-10  2.93321123e-10  1.32903001e-10  1.03767762e-11
  7.44793674e-10  3.13003954e-10  1.42601271e-10  1.11643490e-11
  6.64301728e-10  2.83113952e-10  1.27048228e-10  9.87769425e-12
  8.58084896e-10  3.59594

In [140]:
empirical_cov_matrix = np.cov(noisy_data_vectors, rowvar=False)
frobenius_diff = np.linalg.norm(empirical_cov_matrix[16:,:][:,16:] - compressed_cov[16:,:][:,16:], ord='fro') / np.linalg.norm(compressed_cov[16:,:][:,16:], ord='fro')
print(frobenius_diff)

0.03229405448557699


In [146]:
def save_noisy_compressed_realization(number, noisy_dvs):

    opendata_map3 = fits.open('../data/dv/sim_map3-cosmoCosmogrid-6Nov24-8arcmincut-WITH_COV.fits')
 
    cut_compressed_2pt = noisy_data_vector_cholesky[number][:16]
    for i in range(16,96):
        opendata_map3[1].data[i-16][6] = noisy_data_vector_cholesky[number][i]

    noisy_map3 = '../data/dv/noisy_realizations/sim_map3-NLA-cosmoCosmogrid-0'+str(number+1)+'.fits'
    opendata_map3.writeto(noisy_map3) 
    
    np.savetxt('../data/dv/compressed-moped/sim-2pt-noisy-compressed-0'+str(number+1)+'.txt',cut_compressed_2pt)

In [147]:
for j in range(10,51):
    save_noisy_compressed_realization(j,noisy_data_vectors)

In [154]:
import random

# List of numbers to permute
numbers = [1, 2, 3]

# Generate five random permutations with repetition
random_permutations = [random.sample(numbers, len(numbers)) for _ in range(500)]

# Print the random permutations
for perm in random_permutations[27:32]:
    print(perm)

[2, 3, 1]
[3, 1, 2]
[3, 1, 2]
[2, 3, 1]
[2, 3, 1]
